<font size=6>实验4：注意力机制与Transformer</font>

# 实验简介

本实验介绍使用PyTorch从零开始构建Transformer。在实验中，我们不使用PyTorch内置的Transformer模块，而是基于其架构逐步构建模型。通过本实验，你将掌握以下技能：

- 注意力机制和Transformer模型的基本原理
- 利用PyTorch搭建Transformer模型

# 实验环境准备

首先，你需要安装完成本实验所需的必要组件。

## 安装Miniconda

从以下链接下载Miniconda安装包并完成安装[[Link]](https://mirrors.tuna.tsinghua.edu.cn/anaconda/miniconda/Miniconda3-py310_24.1.2-0-Windows-x86_64.exe)，安装过程中需要选中`Register Anaconda as my default Python 3.10`。若计算机上已经安装好Miniconda或Anaconda，请跳过此步骤。

## 安装Jupyter Notebook

在Anaconda Prompt中执行`pip install notebook`，以安装Jupyter Notebook。

安装完成后，在Anaconda Prompt执行`jupyter notebook`，检查是否能够成功启动Notebook。

若计算机上已经安装好Jupyter Notebook，请跳过此步骤。

## 安装必要的Package

你可以使用以下代码在Jupyter Notebook中检查并安装必要的包，也可事先在Anaconda Prompt中通过`pip`或`conda`安装好。

In [1]:
from importlib.util import find_spec

# 检查PyTorch是否已安装
if find_spec('torch'):
    print('PyTorch is available.')
else:
    !pip install torch -i https://pypi.tuna.tsinghua.edu.cn/simple/

PyTorch is available.


In [2]:
# 防止可能的kmp_duplicate_lib_ok错误
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [3]:
import warnings
warnings.filterwarnings('ignore')

# Transformer模型架构

自2017年的里程碑式论文《Attention is all you need》发表以来，Transformer架构便彻底改变了深度学习的格局。这些架构最初是为神经机器翻译而设计，但它们很快便成为几乎所有前沿自然语言处理系统的基石，并进一步拓展至计算机视觉、音频处理等多个领域。

Transformer需要配置一些参数：

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Dict, Tuple

class TransformerConfig:    
    def __init__(self,
                 src_vocab_size: int = 32000,
                 tgt_vocab_size: int = 32000,
                 d_model: int = 256,
                 num_heads: int = 8,
                 num_encoder_layers: int = 6,
                 num_decoder_layers: int = 6,
                 d_ff: int = 1024,
                 dropout: float = 0.2,
                 max_seq_length: int = 128,
                 pad_token_id: int = 0,
                 shared_embeddings: bool = False):
        """
        src_vocab_size：源语言词汇表大小
        tgt_vocab_size：目标语言词汇表大小
        d_model：模型嵌入向量的维度
        num_heads：注意力头的数量
        num_encoder_layers：编码器层数
        num_decoder_layers：解码器层数
        d_ff：前馈网络的内部维度
        dropout：Dropout比率（正则化丢弃率）
        max_seq_length：最大序列长度
        pad_token_id：填充符（Padding）的Token ID
        shared_embeddings：是否在编码器和解码器之间共享嵌入层
        """
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.d_ff = d_ff
        self.dropout = dropout
        self.max_seq_length = max_seq_length
        self.pad_token_id = pad_token_id
        self.shared_embeddings = shared_embeddings

## 编码器-解码器架构

Transformer架构由简单的组件构成，原论文主要采用的是编码器-解码器架构用于机器翻译。编码器（Encoder）负责处理源语言的信息，而解码器（Decoder）利用编码器提供的上下文信息，将其逐词翻译成目标语言。形式上，编码器将源序列标记 $x_1, \dots, x_n$ 映射为连续表示 $z_1, \dots, z_n$，然后解码器使用该表示一次生成一个目标标记 $y_1, \dots, y_m$。

## 分词

在模型处理文本之前，必须通过分词将其转化为Token。Token是将序列数据转换为块的想法，这是模型可以处理的最小单位。

## 词嵌入

将分词后的序列转换为高维稠密向量（Dense Embeddings）。相比于稀疏的One-Hot编码，稠密嵌入可以捕捉不同概念之间的语义关系，且内存占用更小。
假设词表大小为$V$，模型维度为$d_{model}$，我们学习一个嵌入矩阵$W_e \in \mathbb{R}^{V \times d_{model}}$。

在Transformer编码器中，每个源标记$x^{src}_t$使用源嵌入矩阵映射为稠密向量，并加上位置编码：
$$h^{(0,src)}_t = {W^{src}_e}^\top x^{src}_t + PE_{src}(t)$$

解码器是自回归的。它需要将目标序列向右偏移一位（Shifted Right），并在最前面添加`<sos>`（Start of Sequence）标记。预测时也是如此偏移，这被称为Teacher Forcing。目标向量表示为：
$$g^{(0,tgt)}_{t-1} = {W^{tgt}_e}^\top y^{tgt}_{t-1} + PE_{tgt}(t-1)$$

## 位置编码

由于Transformer是并行处理整个序列的，它本身不具备像RNN那样的序列顺序感知能力。我们需要注入位置信息。原论文使用了一种不需要额外参数、可扩展且连续的方法——正弦位置编码。计算公式如下：
$$PE(p, 2i) = \sin\left(\frac{p}{10000^{\frac{2i}{d_{model}}}}\right)$$
$$PE(p, 2i+1) = \cos\left(\frac{p}{10000^{\frac{2i}{d_{model}}}}\right)$$
其中$p$为位置索引，$i$为维度索引。正弦和余弦函数的波长呈几何级数变化，能够让模型学习相对位置。

位置编码的实现方法如下：

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Dict, Tuple

class PositionalEncoding(nn.Module):
    
    def __init__(self, config: TransformerConfig):
        super(PositionalEncoding, self).__init__()
        
        # 创建位置编码矩阵
        pe = torch.zeros(config.max_seq_length, config.d_model)
        position = torch.arange(0, config.max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, config.d_model, 2).float() * (-math.log(10000.0) / config.d_model))
        
        # 使用正余弦函数
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 添加位置编码
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## 注意力机制

自注意力能够让序列中的每个标记关注序列中的所有其他标记（包括其自身）。

为了实现这一点，我们使用三个矩阵：查询（Query, $Q$）、键（Key, $K$）和值（Value, $V$）。

- $Q$（Query）：我要寻找什么信息？
- $K$（Key）：我能提供什么信息/我的相关性如何？
- $V$（Value）：数据的实际内容。

计算公式为：
$$A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
除以$\sqrt{d_k}$是为了防止点积结果过大导致softmax梯度消失。

In [8]:
def scaled_dot_product_attention(query: torch.Tensor,
                                key: torch.Tensor,
                                value: torch.Tensor,
                                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    计算缩放后的点积注意力
    """
    # 维度
    head_dim = query.size(-1)
    # 计算缩放后的点积
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(head_dim)
    # 需要时使用掩码
    if mask is not None:
        scores = scores.masked_fill(~mask, -1e+30 if scores.dtype == torch.float32 else -1e+4)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output

Transformer还将$Q, K, V$投影到多个头（Heads）中，以便模型在不同的表示子空间中捕捉不同的关系：
$$\text{MultiHead}(H^{(l-1)}) = [\text{head}_1; \dots; \text{head}_h]W^O$$

In [9]:
class MultiHeadAttention(nn.Module):    
    def __init__(self, config: TransformerConfig):
        super(MultiHeadAttention, self).__init__()
        
        self.d_model = config.d_model # 512
        self.num_heads = config.num_heads # 8
        assert self.d_model % self.num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = self.d_model // self.num_heads

        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.k_proj = nn.Linear(self.d_model, self.d_model)
        self.v_proj = nn.Linear(self.d_model, self.d_model)
        
        self.output_proj = nn.Linear(self.d_model, self.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self,
               query: torch.Tensor,
               key: torch.Tensor,
               value: torch.Tensor,
               mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        多头自注意力的前向传播过程
        """
        batch_size = query.size(0)

        # 计算QKV并拆分成多个Head
        
        q = self.q_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        attn_output = scaled_dot_product_attention(q, k, v, mask)
        
        # 将多个Head进行汇总
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.output_proj(attn_output)
        return output

## 层归一化与残差连接

每个子层（注意力和前馈层）之后都有一个残差连接和层归一化（Layer Normalization）：
$$X = \text{LayerNorm}(H^{(l-1)} + \text{Sublayer}(H^{(l-1)}))$$

## 掩码

- 填充掩码 (Padding Mask)：忽略批次中为了对齐长度而添加的`<pad>`标记。
- 因果掩码 (Causal Mask)：在解码器中使用，确保位置$t$的预测只能依赖于小于$t$的已知输出，防止偷看未来的词。

## 前馈神经网络（Feed-Forward Networks）

在每个位置独立应用的两层MLP：
$$FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

In [10]:
class FeedForward(nn.Module):
    """FFN模块"""
    def __init__(self, config: TransformerConfig):
        super(FeedForward, self).__init__()
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        两个全连接层
        """
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x

## 编码器

编码器的主要任务是摄取源序列，并将其转换为一个含上下文信息的连续表示。整个编码器由$N$个完全相同的编码器层堆叠而成。每一层本身不改变张量的形状（始终保持$d_{model}$的维度），使它们可以串联在一起。如果我们追踪一个序列张量$X$穿过单个编码器层，它会经历以下两个主要子层：

1. 多头自注意力：在这个子层中，输入序列中的每个词都会去观察同一个序列中的其他所有词，以此来理解当前的上下文。输入的张量$X$会被用来生成查询（Query）、键（Key）和值（Value）。因为是自注意力，所以$Q = X$，$K = X$，$V = X$。经过多头注意力计算后，我们得到一个聚合了全局上下文的新张量$\text{Sublayer}_1(X)$。

2. 前馈神经网络：在捕捉了词与词之间的关系之后，我们需要对每个位置的词向量进行非线性变换，以提取更高级的特征。计算方式是一个两层的多层感知机，它对序列中的每个位置（Position-wise）独立且完全相同地执行操作：
$$FFN(X') = \max(0, X'W_1 + b_1)W_2 + b_2$$
其中，第一层通常会将维度放大（例如从$d_{model}=512$放大到$d_{ff}=2048$），引入ReLU激活函数后，第二层再将其压缩回$d_{model}$。

在Transformer中，每个子层（包括注意力和前馈网络）之后都紧跟一个Add&Norm操作。Add（残差连接）将子层的输入直接与输出相加。这能够缓解深层网络中的梯度消失问题。Norm（层归一化）对相加后的结果进行归一化，使得特征的均值为0，方差为1，从而稳定训练过程。数学表达为：
$$X' = \text{LayerNorm}(X + \text{SelfAttention}(X))$$

In [11]:
class EncoderLayer(nn.Module):
    """编码器层，包含了self-attention和FFN"""    
    def __init__(self, config: TransformerConfig):
        super(EncoderLayer, self).__init__()
        
        self.self_attn = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        
        self.dropout1 = nn.Dropout(config.dropout)
        self.dropout2 = nn.Dropout(config.dropout)

        self.pad_token_id = config.pad_token_id
        
    def forward(self, 
                x: torch.Tensor,
                src_padding_mask: torch.Tensor) -> torch.Tensor:
        
        """
        x: 输入张量 (batch_size, seq_len, d_model)
        src_mask: 可选的原序列掩码，用于掩盖未来位置
        src_padding_mask: 可选的填充掩码，用于忽略填充token
        """
    
        # 带残差连接和层归一化的自注意力模块
        # 对于编码器，我们将把填充掩码传递给自注意力层
        # 填充掩码的形状将是：(batch_size, 1, 1, seq_len)
        attn_output = self.self_attn(x, x, x, src_padding_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))
        
        return x

In [13]:
class Encoder(nn.Module):
    """编码器，包含多个堆叠的编码器层"""
    def __init__(self, config: TransformerConfig):
        super(Encoder, self).__init__()
        
        self.embedding = nn.Embedding(config.src_vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_encoding = PositionalEncoding(config)
        
        self.layers = nn.ModuleList([EncoderLayer(config) for _ in range(config.num_encoder_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.pad_token_id = config.pad_token_id
        
    def forward(self,
               src: torch.Tensor,
               src_padding_mask: torch.Tensor) -> torch.Tensor:
        x = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)

        for layer in self.layers:
            x = layer(x, src_padding_mask)
        
        x = self.norm(x)
        return x

## 解码器

解码器的任务是根据编码器提供的Memory以及已经生成的历史目标序列，预测下一个词。它同样由$N$个相同的解码器层堆叠而成。解码器层比编码器层更复杂，因为它多了一个子层，用于接收来自编码器的信息。追踪目标张量$Y$穿过单个解码器层，会经历以下三个子层：

1. 带掩码的多头自注意力：解码器在训练时是并行输入整个目标序列的，但它在预测位置$t$的词时，绝对不能提前偷看位置$t+1$及之后的词。在计算注意力分数时，引入因果掩码（Causal Mask/Look-ahead Mask）。掩码作用是将当前词之后的所有词的注意力分数强制设为负无穷大。这样经过Softmax后，未来词的权重就会变成0。

2. 交叉注意力：这是Transformer实现翻译或条件生成的灵魂所在。解码器需要知道在生成当前的词时，我应该重点关注源语言句子中的哪些词。计算方式是一个多头注意力机制，但它的$Q, K, V$来源不同。$Q$（Query）来源于解码器，代表解码器当前的状态和需求；$K, V$来源于编码器，是编码器堆栈的最终输出Memory，代表源序列提供的全局信息。这就好比解码器拿着自己当前翻译到一半的进度（$Q$），去原文的记忆库（$K, V$）里检索接下来最需要的信息。

3. 前馈神经网络：与编码器完全相同，对交叉注意力融合后的特征进行进一步的非线性变换。解码器的最后一层输出后，通常会接一个线性层和Softmax函数，将$d_{model}$维度的向量映射到词表大小的概率分布上，从而选出概率最大的那个词作为最终的预测结果。

In [12]:
class DecoderLayer(nn.Module):
    """解码器层，包含self-attention、cross-attention、FFN"""
    def __init__(self, config: TransformerConfig):
        super(DecoderLayer, self).__init__()
        
        self.self_attn = MultiHeadAttention(config)
        self.cross_attn = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.norm3 = nn.LayerNorm(config.d_model)
        
        self.dropout1 = nn.Dropout(config.dropout)
        self.dropout2 = nn.Dropout(config.dropout)
        self.dropout3 = nn.Dropout(config.dropout)
        
    def forward(self,
               x: torch.Tensor,
               memory: torch.Tensor,
               combined_mask: torch.Tensor,
               memory_padding_mask: torch.Tensor) -> torch.Tensor:
        """
        x: 输入张量 (batch_size, seq_len, d_model)
        memory: 编码器的输出张量 (batch_size, src_seq_len, d_model)
        tgt_padding_mask: 目标序列的填充掩码，用于忽略目标序列中的填充token
        memory_padding_mask: 记忆张量的填充掩码，用于忽略编码器输出中的填充token
        future_mask: 未来位置掩码，用于掩盖未来的位置（防止解码器看到未来信息）
        """
        attn_output = self.self_attn(x, x, x, combined_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        attn_output = self.cross_attn(x, memory, memory, memory_padding_mask)
        x = self.norm2(x + self.dropout2(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))
        
        return x

In [14]:
class Decoder(nn.Module):
    """解码器，包含多个堆叠的解码器层"""
    def __init__(self, config: TransformerConfig):
        super(Decoder, self).__init__()
        
        self.embedding = nn.Embedding(config.tgt_vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_encoding = PositionalEncoding(config)
        
        self.layers = nn.ModuleList([DecoderLayer(config) for _ in range(config.num_decoder_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.pad_token_id = config.pad_token_id
        
    def forward(self,
               tgt: torch.Tensor,
               memory: torch.Tensor,
               tgt_padding_mask: torch.Tensor,
               future_mask: Optional[torch.Tensor],
               memory_padding_mask: torch.Tensor) -> torch.Tensor:
        """
        tgt: 目标张量 (batch_size, tgt_seq_len)
        memory: 编码器输出张量 (batch_size, src_seq_len, d_model)
        tgt_mask: 目标掩码，用于掩盖未来位置（防止解码器看到未来信息）
        memory_mask: 记忆掩码，用于忽略编码器输出中的填充token
        """
        if future_mask is not None:
            combined_mask = self._combine_mask(tgt_padding_mask, future_mask)
        else:
            combined_mask = tgt_padding_mask

        # Embed tokens and add positional encoding
        x = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        
        # Apply decoder layers
        for layer in self.layers:
            x = layer(x, memory, combined_mask, memory_padding_mask)
        
        x = self.norm(x)
        return x

    def _combine_mask(self, tgt_padding_mask: torch.Tensor, future_mask: torch.Tensor) -> torch.Tensor:
        """
        合并填充掩码和未来掩码，用于解码器的自注意力机制。这样做可以让我们只向解码器的自注意力层传递一个合并后的掩码，从而减少传递多个掩码带来的额外开销。
        tgt_padding_mask: 目标序列的填充掩码，形状为 [batch_size, 1, 1, seq_len]
        future_mask: 未来位置掩码（因果掩码），形状为 [seq_len, seq_len]
        """
        batch_size = tgt_padding_mask.size(0)
        seq_len = tgt_padding_mask.size(-1)
        
        padding_mask = tgt_padding_mask.squeeze(2) # [batch_size, 1, seq_len]
        padding_mask = padding_mask.unsqueeze(-1)  # [batch_size, 1, 1, seq_len]
        padding_mask = padding_mask.expand(-1, -1, seq_len, -1)  # [batch_size, 1, seq_len, seq_len]
        
        future_mask = future_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
        future_mask = future_mask.expand(batch_size, 1, -1, -1)  # [batch_size, 1, seq_len, seq_len]
        
        combined_mask = padding_mask & ~future_mask
        
        return combined_mask

## Transformer的完整架构

最终，我们将上述模块进行组合：

In [16]:
class Transformer(nn.Module):
    
    def __init__(self, config: TransformerConfig):
        super(Transformer, self).__init__()
        
        self.config = config
        
        # 创建嵌入层
        if config.shared_embeddings:
            assert config.src_vocab_size == config.tgt_vocab_size, "Vocab sizes must match for shared embeddings"
            self.encoder_embedding = nn.Embedding(config.src_vocab_size, config.d_model, padding_idx=config.pad_token_id)
            self.decoder_embedding = self.encoder_embedding
        else:
            self.encoder_embedding = None
            self.decoder_embedding = None
        
        # Create encoder and decoder
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        
        if config.shared_embeddings:
            self.encoder.embedding = self.encoder_embedding
            self.decoder.embedding = self.decoder_embedding
        
        # Output projection
        self.output_projection = nn.Linear(config.d_model, config.tgt_vocab_size)
        
        # Initialize parameters
        self._reset_parameters()
        
    def forward(self,
               src: torch.Tensor,
               tgt: torch.Tensor,
               future_mask: Optional[torch.Tensor] = None,
               ) -> torch.Tensor:

        src_padding_mask = self._prepare_masks(src)
        tgt_padding_mask = self._prepare_masks(tgt)
        
        # Encode source
        memory = self.encoder(
            src = src, 
            src_padding_mask = src_padding_mask
            )
        
        # Decode target
        output = self.decoder(
            tgt = tgt, 
            memory = memory, 
            tgt_padding_mask = tgt_padding_mask, 
            future_mask = future_mask,
            memory_padding_mask = src_padding_mask
        )
        
        logits = self.output_projection(output)
        
        return logits
    
    def _prepare_masks(self, seq_batch: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mask = (seq_batch != self.config.pad_token_id).unsqueeze(1).unsqueeze(2)
        return mask
    
    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

一些功能函数：

In [17]:
def generate_square_subsequent_mask(size: int, device: torch.device = None) -> torch.Tensor:
    """
    为解码器自注意力中的未来位置生成方形掩码。
    参数：
        size: 序列长度
        device: 创建掩码所用的设备（CPU/GPU）
    返回：
        形状为 [size, size] 的方形掩码张量，其中：
        - False 表示需要关注的位置
        - True 表示需要被掩码掉的位置（即未来位置）
    示例：
        当 size=4 时，生成的掩码如下：
        [[False,  True,  True,  True],
         [False, False,  True,  True],
         [False, False, False,  True],
         [False, False, False, False]]
    """
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    mask = mask.bool()
    if device:
        mask = mask.to(device)
    return mask


def count_parameters(model: nn.Module) -> int:
    """
    计算可学习参数总量
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 一个简单的Transformer模型用例

In [20]:
# 1. 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"正在使用设备: {device}")

# 2. 定义特殊标记并构建词汇表
PAD_TOKEN = "<pad>"  # 填充标记
BOS_TOKEN = "<bos>"  # 序列开始标记
EOS_TOKEN = "<eos>"  # 序列结束标记

# 示例句子
src_sentence_text = "cat sat on the mat"  # 源句子
# 在本演示中，目标句子与源句子相同（简单的复制任务）
# 在实际的翻译任务中，tgt_sentence_text 会是翻译后的句子
tgt_sentence_text = "cat sat on the mat" 

# 构建词汇表 - 方法1：使用set自动去重
all_words = set()
all_words.update([PAD_TOKEN, BOS_TOKEN, EOS_TOKEN])  # 添加特殊标记

for sentence in [src_sentence_text, tgt_sentence_text]:
    for word in sentence.split():
        all_words.add(word)

# 创建单词到ID的映射
word_to_idx = {word: idx for idx, word in enumerate(sorted(all_words))}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
vocab_size = len(word_to_idx)
PAD_ID = word_to_idx[PAD_TOKEN]

print("\n--- 词汇表 ---")
print(f"词汇表大小: {vocab_size}")
print(f"单词到ID的映射: {word_to_idx}")
print(f"填充标记ID: {PAD_ID}")

# 3. 配置Transformer模型
# 为快速演示使用较小的维度
config = TransformerConfig(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=32,                 # 模型嵌入维度
    num_heads=4,                # 注意力头数量
    num_encoder_layers=2,       # 编码器层数
    num_decoder_layers=2,       # 解码器层数
    d_ff=64,                    # 前馈网络维度
    dropout=0.0,                # Dropout比率（设为0以使演示结果确定）
    max_seq_length=15,          # 最大序列长度（必须 >= 实际序列长度）
    pad_token_id=PAD_ID,
    shared_embeddings=True      # 由于此处源/目标词汇表相同，共享嵌入层
)
print("\n--- 模型配置 ---")
for key, value in config.__dict__.items():
    print(f"{key}: {value}")

# 4. 准备数据
# 源句子处理：标记 + EOS + 填充
src_token_list = src_sentence_text.split()
src_ids_temp = [word_to_idx[token] for token in src_token_list]
src_ids_temp.append(word_to_idx[EOS_TOKEN])  # 添加EOS到源序列

# 填充到最大序列长度
padding_needed_src = config.max_seq_length - len(src_ids_temp)
if padding_needed_src < 0:
    raise ValueError(f"源句子太长，超过max_seq_length={config.max_seq_length}")
src_padded_ids = src_ids_temp + [PAD_ID] * padding_needed_src
src_tensor = torch.tensor([src_padded_ids], dtype=torch.long, device=device)

# 目标句子处理（解码器输入）：BOS + 标记 + 填充
tgt_token_list = tgt_sentence_text.split()
tgt_ids_input_temp = [word_to_idx[BOS_TOKEN]]  # 以BOS开始
tgt_ids_input_temp.extend([word_to_idx[token] for token in tgt_token_list])

padding_needed_tgt = config.max_seq_length - len(tgt_ids_input_temp)
if padding_needed_tgt < 0:
    raise ValueError(f"目标句子太长，超过max_seq_length={config.max_seq_length}")
tgt_padded_ids_input = tgt_ids_input_temp + [PAD_ID] * padding_needed_tgt
tgt_tensor_input = torch.tensor([tgt_padded_ids_input], dtype=torch.long, device=device)

print("\n--- 输入张量验证 ---")
print(f"源输入最小ID: {src_tensor.min().item()}, 最大ID: {src_tensor.max().item()}, 词汇表大小: {vocab_size}")
print(f"目标输入最小ID: {tgt_tensor_input.min().item()}, 最大ID: {tgt_tensor_input.max().item()}, 词汇表大小: {vocab_size}")

# 5. 初始化模型
model = Transformer(config).to(device)
model.eval()  # 设置为评估模式（例如，禁用dropout）

# 6. 准备掩码
# 解码器自注意力的未来掩码（因果掩码）：
tgt_seq_len = tgt_tensor_input.size(1)  # 这将是config.max_seq_length
future_mask = generate_square_subsequent_mask(tgt_seq_len, device=device)

# 7. 执行前向传播
# 这模拟了训练过程中的"教师强制"生成或前向传播步骤
# 对于真正的自回归推理，需要在循环中逐个生成标记
print("\n--- 输入张量 ---")
print(f"源输入张量形状: {src_tensor.shape}")
print(f"源输入张量: {src_tensor}")
print(f"目标输入张量（解码器输入）形状: {tgt_tensor_input.shape}")
print(f"目标输入张量（解码器输入）: {tgt_tensor_input}")

with torch.no_grad():  # 本演示不需要计算梯度
    logits = model(src=src_tensor, tgt=tgt_tensor_input, future_mask=future_mask)

# 8. 解释输出
print("\n--- 输出 ---")
print(f"logits形状: {logits.shape}")  # 预期: (batch_size, tgt_seq_len, tgt_vocab_size)

# 使用贪婪解码获取预测的标记ID（取argmax）
predicted_ids = torch.argmax(logits, dim=-1)  # 形状: (batch_size, tgt_seq_len)
print(f"预测的标记ID张量（第一个批次）: {predicted_ids[0]}")

# 将预测的ID转换回单词
predicted_words_list = [idx_to_word[idx.item()] for idx in predicted_ids[0]]

print("\n--- 标记化的输入和预测输出 ---")
source_words_with_eos = src_token_list + [EOS_TOKEN]
print(f"源输入: '{' '.join(source_words_with_eos)}'")
print(f"   填充后的ID: {src_tensor[0].tolist()}")

target_input_words = [BOS_TOKEN] + tgt_token_list
print(f"目标输入（解码器）: '{' '.join(target_input_words)}'")
print(f"   填充后的ID: {tgt_tensor_input[0].tolist()}")

expected_output_words_list = tgt_token_list + [EOS_TOKEN]
print(f"期望的输出序列: '{' '.join(expected_output_words_list)}'")
# （预测序列将包含填充到max_seq_length的部分）

print(f"预测的输出序列（贪婪解码）: '{' '.join(predicted_words_list)}'")

print("\n注意：模型尚未训练，因此预测输出很可能是随机的。")
print("本演示展示了数据通过Transformer模型的流程和张量形状。")

# 统计参数量
num_params = count_parameters(model)
print(f"\n模型中可训练的参数总数: {num_params:,}")

print("\n--- 关于实际的自回归生成（此处未实现，感兴趣可以自行实现）: ---")
print("1. 对 `src_tensor` 进行一次编码，从编码器获取 `memory`。")
print("2. 用 `BOS_TOKEN` ID 初始化 `tgt_input`。")
print("3. 循环执行 `max_seq_length` 步：")
print("   a. 将当前的 `tgt_input` 和 `memory` 传递给解码器。")
print("   b. 获取*最后一个*标记位置的logits。")
print("   c. 选择下一个标记ID（例如，argmax或采样）。")
print("   d. 如果是 `EOS_TOKEN` 或达到最大长度，则停止。")
print("   e. 将新的标记ID追加到 `tgt_input` 并重复。")

正在使用设备: cpu

--- 词汇表 ---
词汇表大小: 8
单词到ID的映射: {'<bos>': 0, '<eos>': 1, '<pad>': 2, 'cat': 3, 'mat': 4, 'on': 5, 'sat': 6, 'the': 7}
填充标记ID: 2

--- 模型配置 ---
src_vocab_size: 8
tgt_vocab_size: 8
d_model: 32
num_heads: 4
num_encoder_layers: 2
num_decoder_layers: 2
d_ff: 64
dropout: 0.0
max_seq_length: 15
pad_token_id: 2
shared_embeddings: True

--- 输入张量验证 ---
源输入最小ID: 1, 最大ID: 7, 词汇表大小: 8
目标输入最小ID: 0, 最大ID: 7, 词汇表大小: 8

--- 输入张量 ---
源输入张量形状: torch.Size([1, 15])
源输入张量: tensor([[3, 6, 5, 7, 4, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2]])
目标输入张量（解码器输入）形状: torch.Size([1, 15])
目标输入张量（解码器输入）: tensor([[0, 3, 6, 5, 7, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2]])

--- 输出 ---
logits形状: torch.Size([1, 15, 8])
预测的标记ID张量（第一个批次）: tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0])

--- 标记化的输入和预测输出 ---
源输入: 'cat sat on the mat <eos>'
   填充后的ID: [3, 6, 5, 7, 4, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2]
目标输入（解码器）: '<bos> cat sat on the mat'
   填充后的ID: [0, 3, 6, 5, 7, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2]
期望的输出序列: 'cat sat on the mat <eos>'
预测的输出序列（贪婪解